# 10 — Map MNH Diagnosis Labels to Cancer-Risk Taxonomy

**Issue:** D4.3 (#139)  
**Depends on:** D4.2 (#138)  
**Purpose:** Map MNH `diagnosis_3` values to the 4-class cancer-risk taxonomy used in BCN20000:
Melanoma | Non-melanoma skin cancer | Benign nevus | Other non-cancer / indeterminate.

All decisions for ambiguous labels are documented in `docs/mnh_taxonomy_mapping.md` and `docs/decision_log.md`.

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/rehmaaziz/revela")
MNH_IN  = ROOT / "data/mel_nevus_histo/mnh_filtered.csv"
MNH_OUT = ROOT / "data/mel_nevus_histo/mnh_mapped.csv"

mnh = pd.read_csv(MNH_IN, low_memory=False)
print(f"Rows: {len(mnh):,}")
print("\nAll unique diagnosis_3 values:")
print(mnh["diagnosis_3"].value_counts(dropna=False).to_string())

Rows: 12,500

All unique diagnosis_3 values:
diagnosis_3
Nevus                                                             8050
Melanoma, NOS                                                     2502
Melanoma in situ                                                  1035
Melanoma Invasive                                                  797
Atypical melanocytic neoplasm                                       70
NaN                                                                 26
Epidermal nevus                                                      6
Atypical intraepithelial melanocytic proliferation                   6
Atypical proliferative nodules in congenital melanocytic nevus       4
Lentigo NOS                                                          3
Mucosal melanotic macule                                             1


In [2]:
# Taxonomy map for labels actually present in filtered MNH.
# Decisions for indeterminate / non-nevus benign / collision rows are documented in
# docs/mnh_taxonomy_mapping.md and decision DEC-014 in docs/decision_log.md.
TAXONOMY_MAP = {
    # --- Melanoma ---
    "Melanoma, NOS":     "Melanoma",
    "Melanoma in situ":  "Melanoma",
    "Melanoma Invasive": "Melanoma",

    # --- Benign nevus ---
    "Nevus": "Benign nevus",

    # --- Other non-cancer / indeterminate ---
    # Indeterminate melanocytic lesions: diagnosis_2 = 'Indeterminate melanocytic proliferations'
    "Atypical melanocytic neoplasm":                                  "Other non-cancer / indeterminate",
    "Atypical intraepithelial melanocytic proliferation":             "Other non-cancer / indeterminate",
    "Atypical proliferative nodules in congenital melanocytic nevus": "Other non-cancer / indeterminate",
    # Non-melanocytic / non-nevus benign skin lesions
    "Epidermal nevus":          "Other non-cancer / indeterminate",
    "Lentigo NOS":              "Other non-cancer / indeterminate",
    "Mucosal melanotic macule": "Other non-cancer / indeterminate",
}

mnh["cancer_risk_class"] = mnh["diagnosis_3"].map(TAXONOMY_MAP)

# Collision-lesion handling: diagnosis_3 is NaN; decide by diagnosis_2.
COLLISION_MAP = {
    "Collision - Only benign proliferations":          "Other non-cancer / indeterminate",
    "Collision - At least one malignant proliferation": "Melanoma",
}
mask_nan = mnh["cancer_risk_class"].isna()
mnh.loc[mask_nan, "cancer_risk_class"] = mnh.loc[mask_nan, "diagnosis_2"].map(COLLISION_MAP)

unmapped = mnh[mnh["cancer_risk_class"].isna()]["diagnosis_3"].value_counts(dropna=False)
if len(unmapped) > 0:
    print("UNMAPPED labels remain — decide before proceeding:")
    print(unmapped.to_string())
else:
    print("All labels mapped successfully.")

All labels mapped successfully.


## Decision summary for ambiguous labels

- **Indeterminate melanocytic lesions** (Atypical melanocytic neoplasm, Atypical intraepithelial melanocytic proliferation, Atypical proliferative nodules in congenital nevus — 80 rows) → `Other non-cancer / indeterminate`. Honest representation: diagnosis_2 says these are genuinely indeterminate.
- **Non-nevus benign pigmentations** (Epidermal nevus, Lentigo NOS, Mucosal melanotic macule — 10 rows) → `Other non-cancer / indeterminate`. These are not melanocytic nevi; mapping them as `Benign nevus` would dilute the BCN20000 nevus class.
- **Collision lesions** (diagnosis_3 NaN — 26 rows) → split by diagnosis_2: benign collisions (15) → `Other non-cancer / indeterminate`; malignant collisions (11) → `Melanoma` (conservative — at least one melanoma present).

Full table: `docs/mnh_taxonomy_mapping.md`. Decision record: `DEC-014` in `docs/decision_log.md`.

In [3]:
assert mnh["cancer_risk_class"].isna().sum() == 0, "Unmapped labels remain — FAIL"
print("Assert passed: zero NaN in cancer_risk_class")
print("\nFinal class distribution:")
print(mnh["cancer_risk_class"].value_counts().to_string())

Assert passed: zero NaN in cancer_risk_class

Final class distribution:
cancer_risk_class
Benign nevus                        8050
Melanoma                            4345
Other non-cancer / indeterminate     105


In [4]:
mnh.to_csv(MNH_OUT, index=False)
print(f"Saved: {MNH_OUT}")

Saved: /Users/rehmaaziz/revela/data/mel_nevus_histo/mnh_mapped.csv
